# E3 — Comparativa de experimentos

Tabla y gráficas con todos los resultados del proyecto.

- Lee automáticamente los `metricas.json` guardados por cada experimento v5+
- Incluye resultados históricos (PCN v1-v5, PoinTr v2-v4) hardcodeados
- Genera tabla ordenada por CD-L1 + gráficas de barras interactivas

In [1]:
# ── CELDA 1: Setup ─────────────────────────────────────────────
import os, json, subprocess
from pathlib import Path
from getpass import getpass
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub: ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR,'-q'], capture_output=True)
    del token
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)
subprocess.run(['pip','install','plotly','--quiet'])
print('[OK] listo')

Mounted at /content/drive
Token GitHub: ··········
[OK] listo


In [2]:
# ── CELDA 2: Datos ─────────────────────────────────────────────
#
# HISTORICOS: experimentos sin metricas.json en Drive (anteriores a v5)
# Los experimentos v5+ tienen metricas.json y se leen automáticamente.
#
# Columna 'datos_limpios':
#   True  = FB v2 + Ob v2 (Rocío-cleaned, alineación correcta)
#   False = FB v1 (desalineado) y/o sintético con problemas
#   'parcial' = datos parcialmente limpios o con compensaciones
#
# Nota sobre resultados de Rocío:
#   Sus métricas son 'mejor_val' (CD-L1 sobre validación del entrenamiento),
#   no CD-L1 de test independiente. Son aproximadamente comparables.
#   Marcados con 'rocio_val' en notas para distinguirlos.

HISTORICOS = [
    # ── PCN (Raquel) ───────────────────────────────────────────
    {'version':'PCN v1',   'modelo':'PCN',    'autor':'Raquel', 'fecha':'2026-08-05',
     'datasets':'FB_v1+sint_v1',      'n_pares_total':2428, 'n_test':237,
     'epochs':100, 'best_epoch':97,   'cd_l1':0.0766, 'f_score':0.019,
     'centrar_en_roto':False, 'datos_limpios':False,
     'notas':'FB v1 desalineado · sint plano simple · primer baseline'},

    {'version':'PCN v3',   'modelo':'PCN',    'autor':'Raquel', 'fecha':'2026-08-13',
     'datasets':'FB_v1+sint_v1',      'n_pares_total':2428, 'n_test':237,
     'epochs':400, 'best_epoch':347,  'cd_l1':0.0665, 'f_score':0.024,
     'centrar_en_roto':False, 'datos_limpios':False,
     'notas':'FB v1 desalineado · A100'},

    {'version':'PCN v4',   'modelo':'PCN',    'autor':'Raquel', 'fecha':'2026-08-15',
     'datasets':'FB_v1+sint_v2',      'n_pares_total':2360, 'n_test':236,
     'epochs':500, 'best_epoch':480,  'cd_l1':0.0641, 'f_score':0.0236,
     'centrar_en_roto':False, 'datos_limpios':False,
     'notas':'FB v1 desalineado · sint v2 plano+chip+cuña'},

    {'version':'PCN v5',   'modelo':'PCN',    'autor':'Raquel', 'fecha':'2026-08-15',
     'datasets':'FB_v1+sint_v2',      'n_pares_total':2360, 'n_test':236,
     'epochs':500, 'best_epoch':445,  'cd_l1':0.0630, 'f_score':0.0257,
     'centrar_en_roto':True,  'datos_limpios':'parcial',
     'notas':'FB v1 desalineado · CENTRAR=True compensa parcialmente · mejor PCN Raquel'},

    # ── PCN (Rocío) ────────────────────────────────────────────
    {'version':'PCN v3 (Rocío)', 'modelo':'PCN', 'autor':'Rocío', 'fecha':'2026-08',
     'datasets':'?',              'n_pares_total':None, 'n_test':None,
     'epochs':300, 'best_epoch':300, 'cd_l1':0.0715, 'f_score':None,
     'centrar_en_roto':None, 'datos_limpios':None,
     'notas':'⚠️ val_loss (no CD test) · datos desconocidos · rocio_val'},

    {'version':'PCN v4 (Rocío)', 'modelo':'PCN', 'autor':'Rocío', 'fecha':'2026-08',
     'datasets':'?',              'n_pares_total':None, 'n_test':None,
     'epochs':150, 'best_epoch':150, 'cd_l1':0.0696, 'f_score':None,
     'centrar_en_roto':None, 'datos_limpios':None,
     'notas':'⚠️ val_loss (no CD test) · pretrain · MEJOR PCN Rocío · rocio_val'},

    {'version':'PCN v9/ft (Rocío)', 'modelo':'PCN', 'autor':'Rocío', 'fecha':'2026-08',
     'datasets':'?',               'n_pares_total':None, 'n_test':None,
     'epochs':550, 'best_epoch':None, 'cd_l1':0.0859, 'f_score':None,
     'centrar_en_roto':None, 'datos_limpios':None,
     'notas':'⚠️ val_loss · finetune 550ep · rocio_val'},

    # ── TopNet (Rocío) ─────────────────────────────────────────
    {'version':'TopNet v1 (Rocío)', 'modelo':'TopNet', 'autor':'Rocío', 'fecha':'2026-08',
     'datasets':'?',                'n_pares_total':None, 'n_test':None,
     'epochs':359, 'best_epoch':None, 'cd_l1':0.0795, 'f_score':None,
     'centrar_en_roto':None, 'datos_limpios':None,
     'notas':'⚠️ val_loss · finetune · MEJOR TopNet Rocío · rocio_val'},

    {'version':'TopNet/pretrain (Rocío)', 'modelo':'TopNet', 'autor':'Rocío', 'fecha':'2026-08',
     'datasets':'?',                      'n_pares_total':None, 'n_test':None,
     'epochs':124, 'best_epoch':None,     'cd_l1':0.0805, 'f_score':None,
     'centrar_en_roto':None, 'datos_limpios':None,
     'notas':'⚠️ val_loss · pretrain · rocio_val'},

    # ── PoinTr (Raquel) ────────────────────────────────────────
    {'version':'PoinTr v2','modelo':'PoinTr', 'autor':'Raquel', 'fecha':'2026-08-24',
     'datasets':'FB_v1+sint_filtrado', 'n_pares_total':402, 'n_test':41,
     'epochs':150, 'best_epoch':150,  'cd_l1':0.0533, 'f_score':0.3278,
     'centrar_en_roto':True,  'datos_limpios':False,
     'notas':'⚠️ F-Score INFLADO (CENTRAR=True) · FB v1 desalineado · 30ep contaminadas'},

    {'version':'PoinTr v3','modelo':'PoinTr', 'autor':'Raquel', 'fecha':'2026-08-25',
     'datasets':'FB_v1+sint_filtrado', 'n_pares_total':402, 'n_test':None,
     'epochs':200, 'best_epoch':None, 'cd_l1':None,   'f_score':None,
     'centrar_en_roto':False, 'datos_limpios':False,
     'notas':'Eval pendiente (sesión Colab terminó) · FB v1 desalineado · val_loss=0.1093 ep155'},

    {'version':'PoinTr v4','modelo':'PoinTr', 'autor':'Raquel', 'fecha':'2026-08-25',
     'datasets':'FB_v2',              'n_pares_total':61,  'n_test':7,
     'epochs':300, 'best_epoch':285,  'cd_l1':0.0569, 'f_score':0.0285,
     'centrar_en_roto':False, 'datos_limpios':True,
     'notas':'Primer exp con datos limpios · test=7 (varianza alta) · solo FB v2'},
]

# Leer metricas.json locales (experimentos v5+)
import json, os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
locales = []

for ruta in sorted(Path('.').glob('E3/resultados_*/metricas.json')):
    try:
        d = json.loads(ruta.read_text(encoding='utf-8'))
        d['datasets'] = '+'.join(d['datasets']) if isinstance(d['datasets'],list) else d['datasets']
        d.setdefault('datos_limpios', True)
        d.setdefault('autor', 'Raquel')
        locales.append(d)
        print(f'[OK local] {ruta}: {d["version"]} CD={d.get("cd_l1")}')
    except Exception as e:
        print(f'[WARN] {ruta}: {e}')

drive_res = Path(f'{DRIVE}/Datos_E2_E3/E3/Raquel/resultados')
if drive_res.exists():
    for ruta in sorted(drive_res.glob('*/metricas.json')):
        try:
            d = json.loads(ruta.read_text(encoding='utf-8'))
            d['datasets'] = '+'.join(d['datasets']) if isinstance(d['datasets'],list) else d['datasets']
            d.setdefault('datos_limpios', True)
            d.setdefault('autor', 'Raquel')
            if not any(x['version']==d['version'] for x in locales):
                locales.append(d)
                print(f'[OK Drive] {d["version"]} CD={d.get("cd_l1")}')
        except: pass

versiones_locales = {d['version'] for d in locales}
historicos_filtrados = [h for h in HISTORICOS if h['version'] not in versiones_locales]
TODOS = historicos_filtrados + locales

print(f'\nTotal: {len(TODOS)} experimentos ({len(historicos_filtrados)} históricos + {len(locales)} Drive/local)')
print(f'  — incluye {sum(1 for h in HISTORICOS if h.get("autor")=="Rocío")} experimentos de Rocío (val_loss, no CD test)')

[OK Drive] v5_fb_obj CD=0.032336724882430216
[OK Drive] v5_obj CD=0.03061180161993678

Total: 14 experimentos (12 históricos + 2 Drive/local)
  — incluye 5 experimentos de Rocío (val_loss, no CD test)


In [3]:
# ── CELDA 3: Tabla ─────────────────────────────────────────────
import plotly.graph_objects as go
import numpy as np

ordenados = sorted(TODOS, key=lambda x: (x.get('cd_l1') is None, x.get('cd_l1') or 9999))

def fmt(v, d=4):
    if v is None: return '—'
    if isinstance(v, float): return f'{v:.{d}f}'
    return str(v)

def fmt_limpios(v):
    if v is True:    return '✅'
    if v is False:   return '❌'
    if v == 'parcial': return '⚠️'
    if v is None:    return '?'
    return str(v)

mejorCD = min((x['cd_l1'] for x in ordenados if x.get('cd_l1') is not None), default=None)

cols = {
    'Versión':      [x['version'] for x in ordenados],
    'Autor':        [x.get('autor','?') for x in ordenados],
    'Modelo':       [x['modelo'] for x in ordenados],
    'Datasets':     [str(x.get('datasets','?'))[:20] for x in ordenados],
    'Limpios':      [fmt_limpios(x.get('datos_limpios')) for x in ordenados],
    'Pares':        [fmt(x.get('n_pares_total'),0) for x in ordenados],
    'N test':       [fmt(x.get('n_test'),0) for x in ordenados],
    'Best ep.':     [fmt(x.get('best_epoch'),0) for x in ordenados],
    'CD-L1 ↓':      [fmt(x.get('cd_l1')) for x in ordenados],
    'F-Score ↑':    [fmt(x.get('f_score')) for x in ordenados],
    'Notas':        [str(x.get('notas',''))[:50] for x in ordenados],
}

colores_fila = []
for x in ordenados:
    cd = x.get('cd_l1')
    autor = x.get('autor', '')
    notas = str(x.get('notas', ''))
    es_rocio = autor == 'Rocío' or 'rocio_val' in notas
    if cd is None:
        colores_fila.append('#F5F5F5')
    elif cd == mejorCD:
        colores_fila.append('#C8E6C9')   # verde = mejor
    elif es_rocio:
        colores_fila.append('#F3E5F5')   # lila = Rocío (val_loss)
    elif x.get('datos_limpios') is True:
        colores_fila.append('#E3F2FD')   # azul = datos limpios
    elif x.get('datos_limpios') == 'parcial':
        colores_fila.append('#FFF9C4')   # amarillo = parcial
    else:
        colores_fila.append('#FFEBEE')   # rosa = datos no limpios

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f'<b>{c}</b>' for c in cols.keys()],
        fill_color='#1565C0', font=dict(color='white', size=11),
        align='center', height=28
    ),
    cells=dict(
        values=list(cols.values()),
        fill_color=[colores_fila]*len(cols),
        align=['left','center','center','left','center','right','right','right','right','right','left'],
        font=dict(size=10), height=23
    )
)])
fig.update_layout(
    title='<b>Comparativa experimentos E3</b> — ordenado por CD-L1 (menor = mejor)<br>'
          '<sup>🟩 mejor resultado | 🟦 datos limpios | 🟨 parcial | 🟥 datos con problemas | 🟪 Rocío (val_loss≈CD) | ⬜ sin eval</sup>',
    height=90 + 26*len(ordenados),
    margin=dict(l=10, r=10, t=80, b=10)
)
fig.show()

if mejorCD: print(f'Mejor CD-L1: {mejorCD:.4f}')
print('\n⚠️  Resultados anteriores a PoinTr v4 usan FB v1 (desalineado) — no directamente comparables con v4+')
print('⚠️  PoinTr v2 F-Score inflado por CENTRAR_EN_ROTO=True')
print('🟪  Resultados de Rocío son mejor_val (CD-L1 validación), no CD-L1 test — aprox. comparables')

Mejor CD-L1: 0.0306

⚠️  Resultados anteriores a PoinTr v4 usan FB v1 (desalineado) — no directamente comparables con v4+
⚠️  PoinTr v2 F-Score inflado por CENTRAR_EN_ROTO=True
🟪  Resultados de Rocío son mejor_val (CD-L1 validación), no CD-L1 test — aprox. comparables


In [4]:
# ── CELDA 4: Gráfica de barras CD-L1 ───────────────────────────
import plotly.graph_objects as go

con_cd = [x for x in ordenados if x.get('cd_l1') is not None]

colores_bar = []
for x in con_cd:
    if x['cd_l1'] == mejorCD: colores_bar.append('#2E7D32')
    elif x['modelo'] == 'PCN': colores_bar.append('#F57F17')
    else: colores_bar.append('#1565C0')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[x['version'] for x in con_cd],
    y=[x['cd_l1'] for x in con_cd],
    marker_color=colores_bar,
    text=[f"{x['cd_l1']:.4f}" for x in con_cd],
    textposition='outside',
    name='CD-L1'
))
fig.update_layout(
    title='<b>CD-L1 por experimento</b> (menor = mejor)',
    xaxis_title='Experimento', yaxis_title='CD-L1',
    yaxis=dict(range=[0, max(x['cd_l1'] for x in con_cd)*1.15]),
    height=500, plot_bgcolor='white',
    yaxis_gridcolor='#EEEEEE'
)
fig.show()

In [5]:
# ── CELDA 5: Gráfica CD vs F-Score (scatter) ────────────────────
import plotly.graph_objects as go

con_ambos = [x for x in TODOS if x.get('cd_l1') is not None and x.get('f_score') is not None]

fig = go.Figure()
for modelo, color, sym in [('PCN','#F57F17','circle'), ('PoinTr','#1565C0','diamond')]:
    puntos = [x for x in con_ambos if x['modelo']==modelo]
    if puntos:
        fig.add_trace(go.Scatter(
            x=[x['cd_l1'] for x in puntos],
            y=[x['f_score'] for x in puntos],
            mode='markers+text',
            marker=dict(size=12, color=color, symbol=sym,
                        line=dict(width=1, color='white')),
            text=[x['version'] for x in puntos],
            textposition='top center',
            name=modelo
        ))

fig.update_layout(
    title='<b>CD-L1 vs F-Score</b> — mejor = abajo a la derecha<br>'
          '<sup>⚠️ PoinTr v2 F-Score inflado (CENTRAR_EN_ROTO=True)</sup>',
    xaxis_title='CD-L1 (↓ mejor)',
    yaxis_title='F-Score (↑ mejor)',
    height=500, plot_bgcolor='white',
    xaxis_gridcolor='#EEEEEE', yaxis_gridcolor='#EEEEEE'
)
fig.show()

In [6]:
# ── CELDA 6 (opcional): Exportar tabla a CSV ────────────────────
import csv
from pathlib import Path

ruta_csv = Path('E3/comparativa_experimentos.csv')
campos = ['version','modelo','fecha','datasets','n_pares_total','n_test',
          'epochs','best_epoch','cd_l1','f_score','centrar_en_roto','notas']

with open(ruta_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=campos, extrasaction='ignore')
    w.writeheader()
    for fila in ordenados:
        w.writerow({k: fila.get(k,'') for k in campos})

print(f'CSV guardado: {ruta_csv}')

# Copiar también a Drive
DRIVE = '/content/drive/MyDrive'
import shutil
dst = Path(f'{DRIVE}/Datos_E2_E3/E3/Raquel/comparativa_experimentos.csv')
shutil.copy2(ruta_csv, dst)
print(f'Copiado a Drive: {dst}')

CSV guardado: E3/comparativa_experimentos.csv
Copiado a Drive: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/comparativa_experimentos.csv
